# Part 1: Data preparation

Owner: _(name)_

Goal: a clean, documented data file and a split that everyone shares. Nothing here changes
`data/sp500_daily.csv` without a group decision; this notebook **reports** what is in it and what
was decided.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/OWNER/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
df.tail()

## 1. First look

In [ ]:
df.info()
df.describe().round(2)

## 2. Missing values and duplicates

In [ ]:
print("missing values per column:\n", df.isna().sum())
print("duplicate dates:", df.index.duplicated().sum())
# Trading days only: a gap of more than 4 calendar days is worth a look (holiday weeks are normal, longer is not)
gaps = df.index.to_series().diff().dt.days
print("largest gaps between consecutive rows (days):")
print(gaps.sort_values(ascending=False).head(10))

## 3. Outliers

For a price series an "outlier" is a day with an extreme return, not an extreme price. Large moves
such as March 2020 are real events, not errors: list them, explain them, keep them.

In [ ]:
r = df["log_return"].dropna()
z = (r - r.mean()) / r.std()
extreme = r[z.abs() > 4]
print(len(extreme), "days beyond 4 standard deviations")
extreme.sort_values().head(10), extreme.sort_values().tail(10)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
df["Close"].plot(ax=ax[0], title="S&P 500 close")
r.plot(ax=ax[1], title="daily log return")
plt.tight_layout()

## 4. Cleaning decisions

Write each decision as one line, with the reason. For example:

- No missing values, nothing to impute.
- No duplicate dates.
- Extreme days kept: they are real market events and the models must face them.

## 5. The split (from `src/config.py`)

Time order, no shuffling. Part 2 confirms the dates; this cell only reports the sizes.

In [ ]:
y = common.target_series(df)
train, val, test = common.split(y)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:11s} {part.index.min().date()} to {part.index.max().date()}  {len(part):5d} days")